# Componentes candidatos del IVLR
## Construcción y auditoría de ingreso, intensidad y educación

Este notebook construye y audita **tres componentes candidatos** para un futuro Índice de Vulnerabilidad Laboral Relativa (IVLR) en el empleo informal:

1. **Ingreso laboral relativo** (posición dentro del trimestre).
2. **Intensidad laboral** (categorías de INTENSI).
3. **Nivel educativo** (código original).

**No se construye un índice agregado, no se asignan pesos ni se definen umbrales de vulnerabilidad.** La estabilidad laboral se excluye por ahora debido a la necesidad de armonizar universos ocupacionales y revisar el tratamiento de trabajadores de casas particulares.

**Alcance:**
- Universo: empleo informal, 2023T4–2025T4 (80.300 observaciones).
- Unidad: persona-trimestre.
- No se utilizan transiciones t+1.
- `cluster_final` solo como contexto descriptivo.

# 1. Configuración del entorno (reproducible)

In [ ]:
# %%


import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore")

# -------------------------------------------------
# Función para detectar la raíz del proyecto
# -------------------------------------------------
def encontrar_raiz_proyecto() -> Path:
    """
    Busca la raíz del proyecto subiendo desde el directorio actual
    hasta encontrar la carpeta 'Salidas/07_clustering'.
    Si no la encuentra, usa la variable de entorno TFM_PROJECT_ROOT.
    Si tampoco, falla.
    """
    # Archivo o carpeta que identifica la raíz del proyecto
    # Puedes cambiar 'Salidas/07_clustering' por algo más genérico como 'README.md' o '.git'
    marcador = "Salidas/07_clustering"
    
    # Comenzar desde el directorio actual (donde se ejecuta el notebook)
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / marcador).exists():
            return parent.resolve()
    
    # Si no se encuentra, intentar con variable de entorno
    env_root = os.getenv("TFM_PROJECT_ROOT")
    if env_root is not None:
        root = Path(env_root).expanduser().resolve()
        if root.exists():
            return root
    
    raise FileNotFoundError(
        "No se pudo encontrar la raíz del proyecto. "
        "Asegúrate de ejecutar el notebook desde una carpeta dentro del proyecto, "
        "o define la variable de entorno TFM_PROJECT_ROOT apuntando a la raíz."
    )

PROJECT_ROOT = encontrar_raiz_proyecto()

DATA_REL = Path("Salidas") / "07_clustering" / "base_final_clusters_sin_region.parquet"
OUT_REL = Path("Salidas") / "09_ivlr"

DATA_PATH = PROJECT_ROOT / DATA_REL
OUT_DIR = PROJECT_ROOT / OUT_REL
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f"No se encontró el archivo: {DATA_PATH}"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_PATH:    {DATA_PATH}")
print(f"OUT_DIR:      {OUT_DIR}")


PROJECT_ROOT: C:\Users\tomas\OneDrive\Master Granada\TFM
DATA_PATH:    C:\Users\tomas\OneDrive\Master Granada\TFM\Salidas\07_clustering\base_final_clusters_sin_region.parquet
OUT_DIR:      C:\Users\tomas\OneDrive\Master Granada\TFM\Salidas\09_ivlr


# 2. Carga robusta y validación del universo

In [ ]:
# %%


def cargar_base_componentes(path):
    """Carga la base sin modificarla: Pandas, PyArrow y CSV."""
    errores = []

    try:
        return pd.read_parquet(path), "parquet_pandas"
    except Exception as error:
        errores.append(
            f"pd.read_parquet: {type(error).__name__}: {error}"
        )

    try:
        tabla = pq.read_table(path)
        return tabla.to_pandas(), "parquet_pyarrow"
    except Exception as error:
        errores.append(
            f"pq.read_table: {type(error).__name__}: {error}"
        )

    csv_path = path.with_suffix(".csv")

    if not csv_path.exists():
        raise FileNotFoundError(
            "No se pudo leer el Parquet y no existe el CSV alternativo.\n"
            + "\n".join(errores)
        )

    for error in errores:
        print(f"Aviso de lectura: {error}")

    return pd.read_csv(csv_path), "csv_fallback"

base, fuente_datos = cargar_base_componentes(DATA_PATH)
print(f"Fuente utilizada: {fuente_datos}")
print(f"Dimensiones de la base: {base.shape}")

assert len(base) == 80300, "El número de filas no coincide con lo esperado (80.300)."

# Columnas necesarias
required_cols = ["id_persona", "ANO4", "TRIMESTRE", "PONDERA", "CAT_OCUP",
                 "NIVEL_ED", "INTENSI", "log_P21_real", "cluster_final"]
missing = [c for c in required_cols if c not in base.columns]
if missing:
    raise ValueError(f"Faltan columnas: {missing}")

# Verificar duplicados
dups = base.duplicated(["id_persona", "ANO4", "TRIMESTRE"]).sum()
assert dups == 0, f"Se encontraron {dups} duplicados en la clave persona-periodo."

# Faltantes en variables clave
na_log = base["log_P21_real"].isna().sum()
na_cluster = base["cluster_final"].isna().sum()
assert na_log == 13989, f"Se esperaban 13.989 NA en log_P21_real, pero se encontraron {na_log}."
assert na_cluster == 549, f"Se esperaban 549 NA en cluster_final, pero se encontraron {na_cluster}."

print("Validación del universo completada.")

Fuente utilizada: parquet_pandas
Dimensiones de la base: (80300, 14)
Validación del universo completada.


## 3. Componente 1: Ingreso laboral relativo

Se reconstruye el ingreso real (`P21_real`) a partir del logaritmo, se calcula el percentil del ingreso dentro de cada año-trimestre (usando `rank(pct=True)` con `method='average'`), y se define el componente como `1 - percentil`.

Interpretación: valores más altos indican mayor vulnerabilidad económica relativa. Los NA se conservan.

In [3]:
# %%
base_ivlr = base.copy()
base_ivlr["P21_real"] = np.expm1(base_ivlr["log_P21_real"])

# Percentil de ingreso por trimestre (incluyendo ceros reales, excluyendo NA)
base_ivlr["percentil_ingreso"] = (
    base_ivlr
    .groupby(["ANO4", "TRIMESTRE"])["P21_real"]
    .rank(method="average", pct=True, na_option="keep")
)

base_ivlr["componente_ingreso"] = 1 - base_ivlr["percentil_ingreso"]

# Verificar rango y NA (el máximo puede ser ligeramente menor que 1)
assert base_ivlr["componente_ingreso"].min(skipna=True) >= 0
assert base_ivlr["componente_ingreso"].max(skipna=True) <= 1
assert base_ivlr["componente_ingreso"].isna().sum() == 13989

print("Componente ingreso construido correctamente.")

Componente ingreso construido correctamente.


## 4. Componente 2: Intensidad laboral

Se crean variables binarias para cada categoría de `INTENSI` (Subocupado, Ocupado pleno, Sobreocupado, No trabajó). Cada observación debe tener exactamente un 1.

No se asigna un puntaje ordinal ni se combinan en un único componente.

In [4]:
# %%
# Convertir INTENSI a string sin perder NA
base_ivlr["INTENSI_str"] = base_ivlr["INTENSI"].astype("string").str.strip()

# Validar categorías
CATEGORIAS_INTENSI = {"Subocupado", "Ocupado pleno", "Sobreocupado", "No trabajó"}
categorias_observadas = set(base_ivlr["INTENSI_str"].dropna().unique())
assert categorias_observadas == CATEGORIAS_INTENSI, (
    f"Las categorías de INTENSI no coinciden: {categorias_observadas}"
)
assert base_ivlr["INTENSI_str"].isna().sum() == 0

base_ivlr["int_subocupado"] = (base_ivlr["INTENSI_str"] == "Subocupado").astype("Int64")
base_ivlr["int_ocupado_pleno"] = (base_ivlr["INTENSI_str"] == "Ocupado pleno").astype("Int64")
base_ivlr["int_sobreocupado"] = (base_ivlr["INTENSI_str"] == "Sobreocupado").astype("Int64")
base_ivlr["int_no_trabajo"] = (base_ivlr["INTENSI_str"] == "No trabajó").astype("Int64")

# Verificar que suman 1 para todas las filas
columnas_intensidad = [
    "int_subocupado",
    "int_ocupado_pleno",
    "int_sobreocupado",
    "int_no_trabajo",
]
suma_intensidad = base_ivlr[columnas_intensidad].sum(axis=1)
assert suma_intensidad.eq(1).all(), "La suma de los indicadores de intensidad no es 1 para todas las filas."

print("Variables de intensidad creadas y verificadas.")

Variables de intensidad creadas y verificadas.


### 4.1 Especificaciones candidatas de insuficiencia laboral

La subocupación se interpreta como insuficiencia laboral. La sobreocupación no se clasifica automáticamente como vulnerabilidad porque puede responder tanto a necesidad económica como a una mayor intensidad de actividad. La categoría “No trabajó” se mantiene como indeterminada en la especificación principal, ya que la persona continúa clasificada como ocupada y la ausencia semanal puede responder a causas heterogéneas.

Se construyen tres especificaciones para análisis de sensibilidad, sin seleccionar todavía una fórmula definitiva ni incorporarlas a un IVLR agregado.

In [16]:
# Especificaciones candidatas de insuficiencia laboral.
# No constituyen todavía un componente definitivo del IVLR.

base_ivlr["intensidad_principal"] = pd.Series(
    pd.NA,
    index=base_ivlr.index,
    dtype="Float64",
)

base_ivlr.loc[
    base_ivlr["INTENSI_str"].eq("Subocupado"),
    "intensidad_principal",
] = 1.0

base_ivlr.loc[
    base_ivlr["INTENSI_str"].isin(
        ["Ocupado pleno", "Sobreocupado"]
    ),
    "intensidad_principal",
] = 0.0


base_ivlr["intensidad_sensibilidad_a"] = (
    base_ivlr["INTENSI_str"]
    .isin(["Subocupado", "No trabajó"])
    .astype("Int64")
)


auditoria_intensidad = pd.DataFrame([
    {
        "especificacion": "principal",
        "definicion": (
            "Subocupado=1; pleno/sobreocupado=0; "
            "no trabajó=NA"
        ),
        "n_observado": int(
            base_ivlr["intensidad_principal"]
            .notna()
            .sum()
        ),
        "n_faltante_o_indeterminado": int(
            base_ivlr["intensidad_principal"]
            .isna()
            .sum()
        ),
        "media": float(
            base_ivlr["intensidad_principal"]
            .mean()
        ),
        "estado": "especificación candidata principal",
    },
    {
        "especificacion": "sensibilidad_a",
        "definicion": (
            "Subocupado/no trabajó=1; "
            "pleno/sobreocupado=0"
        ),
        "n_observado": int(
            base_ivlr["intensidad_sensibilidad_a"]
            .notna()
            .sum()
        ),
        "n_faltante_o_indeterminado": int(
            base_ivlr["intensidad_sensibilidad_a"]
            .isna()
            .sum()
        ),
        "media": float(
            base_ivlr["intensidad_sensibilidad_a"]
            .mean()
        ),
        "estado": "análisis de sensibilidad",
    },
])

display(auditoria_intensidad)

auditoria_intensidad.to_csv(
    OUT_DIR / "auditoria_especificaciones_intensidad.csv",
    index=False,
    encoding="utf-8-sig",
)

,especificacion,definicion,n_observado,n_faltante_o_indeterminado,media,estado
0,principal,Subocupado=1; pleno/sobreocupado=0; no trabajó=NA,78986,1314,0.201770,especificación candidata principal
1,sensibilidad_a,Subocupado/no trabajó=1; pleno/sobreocupado=0,80300,0,0.214832,análisis de sensibilidad


## 5. Componente 3: Educación

Se conserva el código original de `NIVEL_ED` sin recodificar. Se genera una tabla de distribución (absoluta, porcentajes no ponderados y ponderados) y se verifica la cobertura por año-trimestre.

Para la auditoría numérica se crea una versión auxiliar (`educacion_codigo_numerico`), pero la variable original no se modifica.

In [15]:
# %%
base_ivlr["educacion_codigo_original"] = base_ivlr["NIVEL_ED"]
base_ivlr["educacion_codigo_numerico"] = pd.to_numeric(
    base_ivlr["educacion_codigo_original"],
    errors="coerce",
)

# Distribución de NIVEL_ED
dist_educ = (
    base_ivlr
    .groupby(
        [
            "educacion_codigo_original",
            "educacion_etiqueta",
        ],
        dropna=False,
        observed=True,
    )
    .agg(
        n=("educacion_codigo_original", "size"),
        n_ponderado=("PONDERA", "sum"),
    )
    .reset_index()
)

dist_educ["pct_no_ponderado"] = (
    100 * dist_educ["n"] / len(base_ivlr)
)

dist_educ["pct_ponderado"] = (
    100
    * dist_educ["n_ponderado"]
    / base_ivlr["PONDERA"].sum()
)

print("Distribución de NIVEL_ED:")
display(dist_educ)

# Guardar tabla
dist_educ.to_csv(OUT_DIR / "distribucion_nivel_ed.csv", index=False, encoding="utf-8-sig")

Distribución de NIVEL_ED:


,educacion_codigo_original,educacion_etiqueta,n,n_ponderado,pct_no_ponderado,pct_ponderado
0,1,"Primaria incompleta, incluye educación especial",3650,2182603,4.545455,4.287781
1,2,Primaria completa,12820,8153186,15.965131,16.017150
2,3,Secundaria incompleta,19100,12063679,23.785803,23.699417
3,4,Secundaria completa,26138,16242033,32.550436,31.907904
4,5,Superior o universitaria incompleta,11125,6968461,13.854296,13.689726
5,6,Superior o universitaria completa,7173,5105075,8.932752,10.029055
6,7,Sin instrucción,294,187814,0.366127,0.368966


In [10]:
# Etiquetas descriptivas para variables categóricas.
# Se conservan también los códigos originales.

ETIQUETAS_NIVEL_ED = {
    1: "Primaria incompleta, incluye educación especial",
    2: "Primaria completa",
    3: "Secundaria incompleta",
    4: "Secundaria completa",
    5: "Superior o universitaria incompleta",
    6: "Superior o universitaria completa",
    7: "Sin instrucción",
}

ETIQUETAS_CAT_OCUP = {
    1: "Patrón",
    2: "Cuentapropista",
    3: "Obrero o empleado",
    4: "Trabajador familiar sin remuneración",
}

ETIQUETAS_CLUSTER = {
    1: "Cuentapropismo predominantemente unipersonal",
    2: "Asalariados informales jóvenes vinculados al comercio",
    3: "Asalariados informales en establecimientos grandes",
    4: "Asalariados informales en hogares particulares",
    5: "Inserciones informales mixtas en pequeños establecimientos",
}

base_ivlr["educacion_etiqueta"] = (
    pd.to_numeric(
        base_ivlr["educacion_codigo_original"],
        errors="coerce",
    )
    .map(ETIQUETAS_NIVEL_ED)
    .astype("string")
)

base_ivlr["cat_ocup_etiqueta"] = (
    pd.to_numeric(
        base_ivlr["CAT_OCUP"],
        errors="coerce",
    )
    .map(ETIQUETAS_CAT_OCUP)
    .astype("string")
)

base_ivlr["cluster_etiqueta"] = (
    pd.to_numeric(
        base_ivlr["cluster_final"],
        errors="coerce",
    )
    .map(ETIQUETAS_CLUSTER)
    .astype("string")
)

assert base_ivlr["educacion_etiqueta"].isna().sum() == 0
assert (
    base_ivlr["cluster_etiqueta"].isna().sum()
    == base_ivlr["cluster_final"].isna().sum()
    == 549
)

In [17]:
# %% 
# Análisis de NIVEL_ED por grupos de edad.
# Objetivo: evaluar si la futura privación educativa requiere reglas por edad.

# Verificar disponibilidad y validez de la edad
assert "CH06" in base_ivlr.columns, "CH06 no está disponible en base_ivlr."

base_ivlr["edad"] = pd.to_numeric(
    base_ivlr["CH06"],
    errors="coerce",
)

assert base_ivlr["edad"].notna().all(), (
    "Existen edades faltantes o no numéricas."
)

assert base_ivlr["edad"].between(10, 110).all(), (
    "Se detectaron edades fuera del rango esperado."
)

# Grupos de edad analíticos
bins_edad = [10, 18, 25, 35, 45, 55, 65, np.inf]

etiquetas_edad = [
    "10-17",
    "18-24",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65 o más",
]

base_ivlr["grupo_edad"] = pd.cut(
    base_ivlr["edad"],
    bins=bins_edad,
    labels=etiquetas_edad,
    right=False,
    include_lowest=True,
    ordered=True,
)

assert base_ivlr["grupo_edad"].notna().all(), (
    "Alguna edad no pudo asignarse a un grupo."
)

# Tabla no ponderada
tabla_educacion_edad = (
    base_ivlr
    .groupby(
        [
            "grupo_edad",
            "educacion_codigo_original",
            "educacion_etiqueta",
        ],
        observed=True,
        dropna=False,
    )
    .agg(
        n=("id_persona", "size"),
        n_ponderado=("PONDERA", "sum"),
    )
    .reset_index()
)

# Denominadores dentro de cada grupo de edad
tabla_educacion_edad["total_grupo"] = (
    tabla_educacion_edad
    .groupby("grupo_edad", observed=True)["n"]
    .transform("sum")
)

tabla_educacion_edad["total_ponderado_grupo"] = (
    tabla_educacion_edad
    .groupby(
        "grupo_edad",
        observed=True,
    )["n_ponderado"]
    .transform("sum")
)

tabla_educacion_edad["pct_no_ponderado"] = (
    100
    * tabla_educacion_edad["n"]
    / tabla_educacion_edad["total_grupo"]
)

tabla_educacion_edad["pct_ponderado"] = (
    100
    * tabla_educacion_edad["n_ponderado"]
    / tabla_educacion_edad["total_ponderado_grupo"]
)

tabla_educacion_edad[
    ["pct_no_ponderado", "pct_ponderado"]
] = tabla_educacion_edad[
    ["pct_no_ponderado", "pct_ponderado"]
].round(2)

display(tabla_educacion_edad)

tabla_educacion_edad.to_csv(
    OUT_DIR / "nivel_ed_por_grupo_edad.csv",
    index=False,
    encoding="utf-8-sig",
)

# Resumen analítico de educación no completada
# No constituye todavía un componente del IVLR.

base_ivlr["educacion_hasta_secundaria_incompleta"] = (
    base_ivlr["educacion_codigo_numerico"]
    .isin([1, 2, 3, 7])
    .astype("Int64")
)

resumen_educacion_edad = (
    base_ivlr
    .groupby(
        "grupo_edad",
        observed=True,
    )
    .agg(
        n=("id_persona", "size"),
        edad_media=("edad", "mean"),
        n_hasta_secundaria_incompleta=(
            "educacion_hasta_secundaria_incompleta",
            "sum",
        ),
        proporcion_hasta_secundaria_incompleta=(
            "educacion_hasta_secundaria_incompleta",
            "mean",
        ),
        poblacion_ponderada=("PONDERA", "sum"),
    )
    .reset_index()
)

# Cálculo ponderado por grupo
ponderado_educacion = (
    base_ivlr
    .assign(
        privacion_x_ponderador=(
            base_ivlr[
                "educacion_hasta_secundaria_incompleta"
            ]
            * base_ivlr["PONDERA"]
        )
    )
    .groupby(
        "grupo_edad",
        observed=True,
    )
    .agg(
        privacion_ponderada=(
            "privacion_x_ponderador",
            "sum",
        ),
        total_ponderado=("PONDERA", "sum"),
    )
    .reset_index()
)

ponderado_educacion[
    "proporcion_ponderada_hasta_secundaria_incompleta"
] = (
    ponderado_educacion["privacion_ponderada"]
    / ponderado_educacion["total_ponderado"]
)

resumen_educacion_edad = resumen_educacion_edad.merge(
    ponderado_educacion[
        [
            "grupo_edad",
            "proporcion_ponderada_hasta_secundaria_incompleta",
        ]
    ],
    on="grupo_edad",
    how="left",
    validate="1:1",
)

resumen_educacion_edad[
    [
        "edad_media",
        "proporcion_hasta_secundaria_incompleta",
        "proporcion_ponderada_hasta_secundaria_incompleta",
    ]
] = resumen_educacion_edad[
    [
        "edad_media",
        "proporcion_hasta_secundaria_incompleta",
        "proporcion_ponderada_hasta_secundaria_incompleta",
    ]
].round(4)

display(resumen_educacion_edad)

resumen_educacion_edad.to_csv(
    OUT_DIR / "resumen_educacion_por_grupo_edad.csv",
    index=False,
    encoding="utf-8-sig",
)

# Controles
assert tabla_educacion_edad["n"].sum() == 80300
assert resumen_educacion_edad["n"].sum() == 80300

print(
    "Análisis educativo por edad completado. "
    "No se creó todavía un puntaje educativo."
)

,grupo_edad,educacion_codigo_original,educacion_etiqueta,n,n_ponderado,total_grupo,total_ponderado_grupo,pct_no_ponderado,pct_ponderado
0,10-17,1,"Primaria incompleta, incluye educación especial",29,23574,567,505985,5.11,4.66
1,10-17,2,Primaria completa,29,13472,567,505985,5.11,2.66
2,10-17,3,Secundaria incompleta,473,435510,567,505985,83.42,86.07
3,10-17,4,Secundaria completa,27,22714,567,505985,4.76,4.49
4,10-17,5,Superior o universitaria incompleta,7,8353,567,505985,1.23,1.65
5,10-17,7,Sin instrucción,2,2362,567,505985,0.35,0.47
6,18-24,1,"Primaria incompleta, incluye educación especial",180,90678,13101,8161135,1.37,1.11
7,18-24,2,Primaria completa,534,316894,13101,8161135,4.08,3.88
8,18-24,3,Secundaria incompleta,3794,2314570,13101,8161135,28.96,28.36
9,18-24,4,Secundaria completa,5046,3087519,13101,8161135,38.52,37.83


,grupo_edad,n,edad_media,n_hasta_secundaria_incompleta,proporcion_hasta_secundaria_incompleta,poblacion_ponderada,proporcion_ponderada_hasta_secundaria_incompleta
0,10-17,567,16.0441,533,0.94,505985,0.9386
1,18-24,13101,21.5264,4520,0.345,8161135,0.3346
2,25-34,21896,29.3168,7646,0.3492,13272072,0.3566
3,35-44,17452,39.4079,7749,0.444,11331481,0.4289
4,45-54,14134,49.1946,7488,0.5298,8947862,0.5272
5,55-64,9109,59.1114,5506,0.6045,5787240,0.596
6,65 o más,4041,69.9136,2422,0.5994,2897076,0.5599


Análisis educativo por edad completado. No se creó todavía un puntaje educativo.


## Especificaciones candidatas del componente educativo

La educación se considera un recurso individual asociado con la capacidad de adaptación laboral, pero no una condición del puesto de trabajo. Por ello, no se aplica un umbral uniforme a todas las edades.

La especificación principal utiliza reglas diferenciadas:

- 10–17 años: no evaluable, porque la trayectoria educativa se encuentra en curso.
- 18–24 años: privación educativa únicamente para primaria incompleta, primaria completa o sin instrucción.
- 25 años o más: privación educativa para primaria incompleta, primaria completa, secundaria incompleta o sin instrucción.

Como análisis de sensibilidad, se construye una segunda especificación disponible únicamente para personas de 25 años o más, donde se considera privación tener hasta secundaria incompleta o no contar con instrucción.

Estas variables son candidatas y todavía no se incorporan a un IVLR agregado.

In [21]:
# Especificaciones candidatas del componente educativo.
# 1 = privación educativa
# 0 = sin privación según la regla
# NA = no evaluable

nivel_ed_num = pd.to_numeric(
    base_ivlr["educacion_codigo_original"],
    errors="coerce",
)

edad_num = pd.to_numeric(
    base_ivlr["edad"],
    errors="coerce",
)

assert nivel_ed_num.notna().all()
assert edad_num.notna().all()


# Especificación principal diferenciada por edad
base_ivlr["educacion_principal"] = pd.Series(
    pd.NA,
    index=base_ivlr.index,
    dtype="Float64",
)

# 18–24 años:
# privación severa = primaria incompleta, primaria completa o sin instrucción
mask_18_24 = edad_num.between(18, 24)

base_ivlr.loc[
    mask_18_24,
    "educacion_principal",
] = (
    nivel_ed_num.loc[mask_18_24]
    .isin([1, 2, 7])
    .astype(float)
)

# 25 años o más:
# privación = hasta secundaria incompleta o sin instrucción
mask_25_mas = edad_num.ge(25)

base_ivlr.loc[
    mask_25_mas,
    "educacion_principal",
] = (
    nivel_ed_num.loc[mask_25_mas]
    .isin([1, 2, 3, 7])
    .astype(float)
)

# Menores de 18 años permanecen como NA
assert base_ivlr.loc[
    edad_num.lt(18),
    "educacion_principal",
].isna().all()


# Sensibilidad:
# disponible únicamente para personas de 25 años o más
base_ivlr["educacion_sensibilidad_25_mas"] = pd.Series(
    pd.NA,
    index=base_ivlr.index,
    dtype="Float64",
)

base_ivlr.loc[
    mask_25_mas,
    "educacion_sensibilidad_25_mas",
] = (
    nivel_ed_num.loc[mask_25_mas]
    .isin([1, 2, 3, 7])
    .astype(float)
)


# Auditoría de las dos especificaciones
auditoria_educacion = pd.DataFrame([
    {
        "especificacion": "principal_diferenciada_por_edad",
        "definicion": (
            "10-17=NA; 18-24: códigos 1,2,7=1; "
            "25 o más: códigos 1,2,3,7=1"
        ),
        "n_total": len(base_ivlr),
        "n_observado": int(
            base_ivlr["educacion_principal"]
            .notna()
            .sum()
        ),
        "n_no_evaluable": int(
            base_ivlr["educacion_principal"]
            .isna()
            .sum()
        ),
        "n_privacion": int(
            base_ivlr["educacion_principal"]
            .sum(skipna=True)
        ),
        "proporcion_privacion": float(
            base_ivlr["educacion_principal"]
            .mean(skipna=True)
        ),
        "estado": "especificación candidata principal",
    },
    {
        "especificacion": "sensibilidad_25_mas",
        "definicion": (
            "Menores de 25=NA; 25 o más: "
            "códigos 1,2,3,7=1"
        ),
        "n_total": len(base_ivlr),
        "n_observado": int(
            base_ivlr[
                "educacion_sensibilidad_25_mas"
            ]
            .notna()
            .sum()
        ),
        "n_no_evaluable": int(
            base_ivlr[
                "educacion_sensibilidad_25_mas"
            ]
            .isna()
            .sum()
        ),
        "n_privacion": int(
            base_ivlr[
                "educacion_sensibilidad_25_mas"
            ]
            .sum(skipna=True)
        ),
        "proporcion_privacion": float(
            base_ivlr[
                "educacion_sensibilidad_25_mas"
            ]
            .mean(skipna=True)
        ),
        "estado": "análisis de sensibilidad",
    },
])

display(auditoria_educacion)


# Auditoría por grupo de edad
auditoria_educacion_grupo = (
    base_ivlr
    .groupby(
        "grupo_edad",
        observed=True,
        dropna=False,
    )
    .agg(
        n=("id_persona", "size"),
        n_evaluable_principal=(
            "educacion_principal",
            lambda x: x.notna().sum(),
        ),
        n_privacion_principal=(
            "educacion_principal",
            lambda x: x.sum(skipna=True),
        ),
        proporcion_privacion_principal=(
            "educacion_principal",
            lambda x: x.mean(skipna=True),
        ),
        n_evaluable_sensibilidad=(
            "educacion_sensibilidad_25_mas",
            lambda x: x.notna().sum(),
        ),
        proporcion_privacion_sensibilidad=(
            "educacion_sensibilidad_25_mas",
            lambda x: x.mean(skipna=True),
        ),
    )
    .reset_index()
)

display(auditoria_educacion_grupo)


# Controles
assert len(base_ivlr) == 80300

assert (
    base_ivlr["educacion_principal"]
    .dropna()
    .isin([0.0, 1.0])
    .all()
)

assert (
    base_ivlr["educacion_sensibilidad_25_mas"]
    .dropna()
    .isin([0.0, 1.0])
    .all()
)

assert (
    base_ivlr["educacion_principal"]
    .isna()
    .sum()
    == edad_num.lt(18).sum()
)

assert (
    base_ivlr["educacion_sensibilidad_25_mas"]
    .isna()
    .sum()
    == edad_num.lt(25).sum()
)


# Exportaciones nuevas
auditoria_educacion.to_csv(
    OUT_DIR / "auditoria_especificaciones_educacion.csv",
    index=False,
    encoding="utf-8-sig",
)

auditoria_educacion_grupo.to_csv(
    OUT_DIR / "auditoria_educacion_por_grupo_edad.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Especificaciones educativas construidas. "
    "No se creó un IVLR agregado."
)

,especificacion,definicion,n_total,n_observado,n_no_evaluable,n_privacion,proporcion_privacion,estado
0,principal_diferenciada_por_edad,"10-17=NA; 18-24: códigos 1,2,7=1; 25 o más: có...",80300,79733,567,31537,0.395533,especificación candidata principal
1,sensibilidad_25_mas,"Menores de 25=NA; 25 o más: códigos 1,2,3,7=1",80300,66632,13668,30811,0.462405,análisis de sensibilidad


,grupo_edad,n,n_evaluable_principal,n_privacion_principal,proporcion_privacion_principal,n_evaluable_sensibilidad,proporcion_privacion_sensibilidad
0,10-17,567,0,0.0,<NA>,0,<NA>
1,18-24,13101,13101,726.0,0.055416,0,<NA>
2,25-34,21896,21896,7646.0,0.349196,21896,0.349196
3,35-44,17452,17452,7749.0,0.444018,17452,0.444018
4,45-54,14134,14134,7488.0,0.529786,14134,0.529786
5,55-64,9109,9109,5506.0,0.604457,9109,0.604457
6,65 o más,4041,4041,2422.0,0.599357,4041,0.599357


Especificaciones educativas construidas. No se creó un IVLR agregado.


## Análisis de solapamiento entre componentes

Se evalúa si ingreso relativo, insuficiencia laboral y privación educativa aportan información complementaria o si presentan redundancia. El análisis es descriptivo y no selecciona pesos ni construye un IVLR agregado.

Se examinan:

- disponibilidad conjunta;
- correlaciones y asociaciones;
- diferencias del ingreso según intensidad y educación;
- combinaciones de privaciones;
- patrón de ingreso faltante;
- asociación entre ingreso y educación dentro de grupos de edad.

In [22]:
# Análisis descriptivo de solapamiento entre componentes.
# No construye un IVLR agregado.

from scipy.stats import (
    chi2_contingency,
    mannwhitneyu,
    pointbiserialr,
    spearmanr,
)


VARIABLE_INGRESO = "componente_ingreso"
VARIABLE_INTENSIDAD = "intensidad_principal"
VARIABLE_EDUCACION = "educacion_principal"


# ----------------------------------------------------------
# 1. Disponibilidad individual y conjunta
# ----------------------------------------------------------

base_ivlr["disponible_ingreso"] = (
    base_ivlr[VARIABLE_INGRESO].notna()
)

base_ivlr["disponible_intensidad"] = (
    base_ivlr[VARIABLE_INTENSIDAD].notna()
)

base_ivlr["disponible_educacion"] = (
    base_ivlr[VARIABLE_EDUCACION].notna()
)

base_ivlr["componentes_disponibles"] = (
    base_ivlr[
        [
            "disponible_ingreso",
            "disponible_intensidad",
            "disponible_educacion",
        ]
    ]
    .sum(axis=1)
    .astype("Int64")
)

base_ivlr["tres_componentes_disponibles"] = (
    base_ivlr["componentes_disponibles"].eq(3)
)

disponibilidad_componentes = pd.DataFrame([
    {
        "componente": "ingreso",
        "n_observado": int(
            base_ivlr["disponible_ingreso"].sum()
        ),
        "n_faltante": int(
            (~base_ivlr["disponible_ingreso"]).sum()
        ),
        "pct_observado": float(
            100 * base_ivlr["disponible_ingreso"].mean()
        ),
    },
    {
        "componente": "intensidad",
        "n_observado": int(
            base_ivlr["disponible_intensidad"].sum()
        ),
        "n_faltante": int(
            (~base_ivlr["disponible_intensidad"]).sum()
        ),
        "pct_observado": float(
            100 * base_ivlr["disponible_intensidad"].mean()
        ),
    },
    {
        "componente": "educacion",
        "n_observado": int(
            base_ivlr["disponible_educacion"].sum()
        ),
        "n_faltante": int(
            (~base_ivlr["disponible_educacion"]).sum()
        ),
        "pct_observado": float(
            100 * base_ivlr["disponible_educacion"].mean()
        ),
    },
    {
        "componente": "tres_componentes",
        "n_observado": int(
            base_ivlr["tres_componentes_disponibles"].sum()
        ),
        "n_faltante": int(
            (~base_ivlr["tres_componentes_disponibles"]).sum()
        ),
        "pct_observado": float(
            100
            * base_ivlr[
                "tres_componentes_disponibles"
            ].mean()
        ),
    },
])

display(disponibilidad_componentes)


# ----------------------------------------------------------
# 2. Funciones auxiliares
# ----------------------------------------------------------

def cramer_v(x, y):
    datos = pd.DataFrame({
        "x": x,
        "y": y,
    }).dropna()

    if (
        datos.empty
        or datos["x"].nunique() < 2
        or datos["y"].nunique() < 2
    ):
        return np.nan, len(datos)

    tabla = pd.crosstab(
        datos["x"],
        datos["y"],
    )

    chi2, _, _, _ = chi2_contingency(tabla)

    n = tabla.to_numpy().sum()
    filas, columnas = tabla.shape

    denominador = min(
        filas - 1,
        columnas - 1,
    )

    if denominador == 0:
        return np.nan, n

    valor = np.sqrt(
        (chi2 / n) / denominador
    )

    return float(valor), int(n)


def correlacion_spearman(x, y):
    datos = pd.DataFrame({
        "x": x,
        "y": y,
    }).dropna()

    if (
        len(datos) == 0
        or datos["x"].nunique() < 2
        or datos["y"].nunique() < 2
    ):
        return np.nan, np.nan, len(datos)

    resultado = spearmanr(
        datos["x"],
        datos["y"],
    )

    return (
        float(resultado.statistic),
        float(resultado.pvalue),
        len(datos),
    )


def correlacion_punto_biserial(continua, binaria):
    datos = pd.DataFrame({
        "continua": continua,
        "binaria": binaria,
    }).dropna()

    if (
        len(datos) == 0
        or datos["continua"].nunique() < 2
        or datos["binaria"].nunique() < 2
    ):
        return np.nan, np.nan, len(datos)

    resultado = pointbiserialr(
        datos["binaria"].astype(float),
        datos["continua"].astype(float),
    )

    return (
        float(resultado.statistic),
        float(resultado.pvalue),
        len(datos),
    )


# ----------------------------------------------------------
# 3. Asociaciones entre componentes
# ----------------------------------------------------------

resultados_asociacion = []

r, p, n = correlacion_punto_biserial(
    base_ivlr[VARIABLE_INGRESO],
    base_ivlr[VARIABLE_INTENSIDAD],
)

resultados_asociacion.append({
    "par": "ingreso_vs_intensidad",
    "metodo": "correlacion_punto_biserial",
    "estadistico": r,
    "p_valor": p,
    "n_completo": n,
})


r, p, n = correlacion_punto_biserial(
    base_ivlr[VARIABLE_INGRESO],
    base_ivlr[VARIABLE_EDUCACION],
)

resultados_asociacion.append({
    "par": "ingreso_vs_educacion",
    "metodo": "correlacion_punto_biserial",
    "estadistico": r,
    "p_valor": p,
    "n_completo": n,
})


v, n = cramer_v(
    base_ivlr[VARIABLE_INTENSIDAD],
    base_ivlr[VARIABLE_EDUCACION],
)

resultados_asociacion.append({
    "par": "intensidad_vs_educacion",
    "metodo": "V_de_Cramer",
    "estadistico": v,
    "p_valor": np.nan,
    "n_completo": n,
})


rho, p, n = correlacion_spearman(
    base_ivlr[VARIABLE_INGRESO],
    base_ivlr["educacion_codigo_numerico"],
)

resultados_asociacion.append({
    "par": "ingreso_vs_codigo_educativo_original",
    "metodo": "Spearman_descriptivo",
    "estadistico": rho,
    "p_valor": p,
    "n_completo": n,
})


asociaciones_componentes = pd.DataFrame(
    resultados_asociacion
)

display(asociaciones_componentes)


# ----------------------------------------------------------
# 4. Ingreso según intensidad y educación
# ----------------------------------------------------------

ingreso_por_componentes = (
    base_ivlr
    .groupby(
        [
            VARIABLE_INTENSIDAD,
            VARIABLE_EDUCACION,
        ],
        dropna=False,
    )
    .agg(
        n=("id_persona", "size"),
        n_ingreso_observado=(
            VARIABLE_INGRESO,
            lambda x: x.notna().sum(),
        ),
        pct_ingreso_observado=(
            VARIABLE_INGRESO,
            lambda x: 100 * x.notna().mean(),
        ),
        media_vulnerabilidad_ingreso=(
            VARIABLE_INGRESO,
            "mean",
        ),
        mediana_vulnerabilidad_ingreso=(
            VARIABLE_INGRESO,
            "median",
        ),
        ingreso_real_mediano=(
            "P21_real",
            "median",
        ),
    )
    .reset_index()
)

display(ingreso_por_componentes)


# ----------------------------------------------------------
# 5. Combinaciones de privaciones
# ----------------------------------------------------------

mask_combinaciones = (
    base_ivlr[VARIABLE_INTENSIDAD].notna()
    & base_ivlr[VARIABLE_EDUCACION].notna()
)

tabla_combinaciones = (
    base_ivlr
    .loc[mask_combinaciones]
    .groupby(
        [
            VARIABLE_INTENSIDAD,
            VARIABLE_EDUCACION,
        ],
        dropna=False,
    )
    .agg(
        n=("id_persona", "size"),
        poblacion_ponderada=("PONDERA", "sum"),
        n_ingreso_observado=(
            VARIABLE_INGRESO,
            lambda x: x.notna().sum(),
        ),
        media_componente_ingreso=(
            VARIABLE_INGRESO,
            "mean",
        ),
    )
    .reset_index()
)

tabla_combinaciones["pct_no_ponderado"] = (
    100
    * tabla_combinaciones["n"]
    / tabla_combinaciones["n"].sum()
)

tabla_combinaciones["pct_ponderado"] = (
    100
    * tabla_combinaciones["poblacion_ponderada"]
    / tabla_combinaciones["poblacion_ponderada"].sum()
)

display(tabla_combinaciones)


# ----------------------------------------------------------
# 6. Patrón de ingreso faltante
# ----------------------------------------------------------

base_ivlr["ingreso_faltante"] = (
    base_ivlr[VARIABLE_INGRESO].isna()
    .astype("Int64")
)

faltante_ingreso_componentes = (
    base_ivlr
    .groupby(
        [
            VARIABLE_INTENSIDAD,
            VARIABLE_EDUCACION,
        ],
        dropna=False,
    )
    .agg(
        n=("id_persona", "size"),
        n_ingreso_faltante=(
            "ingreso_faltante",
            "sum",
        ),
        pct_ingreso_faltante=(
            "ingreso_faltante",
            lambda x: 100 * x.mean(),
        ),
    )
    .reset_index()
)

display(faltante_ingreso_componentes)


# ----------------------------------------------------------
# 7. Ingreso y educación dentro de grupos de edad
# ----------------------------------------------------------

asociacion_ingreso_educacion_edad = []

for grupo, datos in base_ivlr.groupby(
    "grupo_edad",
    observed=True,
    dropna=False,
):
    rho, p, n = correlacion_spearman(
        datos[VARIABLE_INGRESO],
        datos["educacion_codigo_numerico"],
    )

    asociacion_ingreso_educacion_edad.append({
        "grupo_edad": grupo,
        "rho_spearman": rho,
        "p_valor": p,
        "n_completo": n,
    })

asociacion_ingreso_educacion_edad = pd.DataFrame(
    asociacion_ingreso_educacion_edad
)

display(asociacion_ingreso_educacion_edad)


# ----------------------------------------------------------
# 8. Tabla de casos completos por periodo y cluster
# ----------------------------------------------------------

casos_completos_contexto = (
    base_ivlr
    .groupby(
        [
            "ANO4",
            "TRIMESTRE",
            "cluster_final",
        ],
        dropna=False,
    )
    .agg(
        n=("id_persona", "size"),
        n_tres_componentes=(
            "tres_componentes_disponibles",
            "sum",
        ),
        pct_tres_componentes=(
            "tres_componentes_disponibles",
            lambda x: 100 * x.mean(),
        ),
    )
    .reset_index()
)

display(casos_completos_contexto)


# ----------------------------------------------------------
# 9. Controles
# ----------------------------------------------------------

assert len(base_ivlr) == 80300

assert (
    disponibilidad_componentes.loc[
        disponibilidad_componentes["componente"]
        == "ingreso",
        "n_faltante",
    ].iloc[0]
    == 13989
)

assert (
    tabla_combinaciones["n"].sum()
    == int(mask_combinaciones.sum())
)

assert (
    faltante_ingreso_componentes["n"].sum()
    == 80300
)


# ----------------------------------------------------------
# 10. Exportaciones
# ----------------------------------------------------------

disponibilidad_componentes.to_csv(
    OUT_DIR / "disponibilidad_conjunta_componentes.csv",
    index=False,
    encoding="utf-8-sig",
)

asociaciones_componentes.to_csv(
    OUT_DIR / "asociaciones_componentes_ivlr.csv",
    index=False,
    encoding="utf-8-sig",
)

ingreso_por_componentes.to_csv(
    OUT_DIR / "ingreso_por_intensidad_educacion.csv",
    index=False,
    encoding="utf-8-sig",
)

tabla_combinaciones.to_csv(
    OUT_DIR / "combinaciones_privaciones_ivlr.csv",
    index=False,
    encoding="utf-8-sig",
)

faltante_ingreso_componentes.to_csv(
    OUT_DIR / "faltante_ingreso_por_componentes.csv",
    index=False,
    encoding="utf-8-sig",
)

asociacion_ingreso_educacion_edad.to_csv(
    OUT_DIR / "asociacion_ingreso_educacion_por_edad.csv",
    index=False,
    encoding="utf-8-sig",
)

casos_completos_contexto.to_csv(
    OUT_DIR / "casos_completos_componentes_contexto.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Análisis de solapamiento completado. "
    "No se construyó un IVLR agregado."
)

,componente,n_observado,n_faltante,pct_observado
0,ingreso,66311,13989,82.579078
1,intensidad,78986,1314,98.363636
2,educacion,79733,567,99.293898
3,tres_componentes,64761,15539,80.648817


,par,metodo,estadistico,p_valor,n_completo
0,ingreso_vs_intensidad,correlacion_punto_biserial,0.292329,0.000000e+00,65222
1,ingreso_vs_educacion,correlacion_punto_biserial,0.043569,4.828472e-29,65846
2,intensidad_vs_educacion,V_de_Cramer,0.004779,NaN,78424
3,ingreso_vs_codigo_educativo_original,Spearman_descriptivo,-0.082587,1.065587e-100,66311


,intensidad_principal,educacion_principal,n,n_ingreso_observado,pct_ingreso_observado,media_vulnerabilidad_ingreso,mediana_vulnerabilidad_ingreso,ingreso_real_mediano
0,0.0,0.0,37940,31147,82.095414,0.444877,0.422836,400000.000000
1,0.0,1.0,24653,20466,83.016266,0.469739,0.440065,367949.744954
2,0.0,<NA>,456,371,81.359649,0.721429,0.825052,129566.801834
3,1.0,0.0,9503,7863,82.742292,0.656651,0.716564,200552.811997
4,1.0,1.0,6328,5285,83.517699,0.677375,0.746716,180497.530797
5,1.0,<NA>,106,90,84.905660,0.808032,0.855614,103272.212025
6,<NA>,0.0,753,626,83.134130,0.518772,0.521147,304553.594472
7,<NA>,1.0,556,459,82.553957,0.587826,0.639153,245630.676009
8,<NA>,<NA>,5,4,80.000000,0.659414,0.811965,135578.496557


,intensidad_principal,educacion_principal,n,poblacion_ponderada,n_ingreso_observado,media_componente_ingreso,pct_no_ponderado,pct_ponderado
0,0.0,0.0,37940,23664765,31147,0.444877,48.378048,47.996493
1,0.0,1.0,24653,14821635,20466,0.469739,31.435530,30.061000
2,1.0,0.0,9503,6340162,7863,0.656651,12.117464,12.859014
3,1.0,1.0,6328,4478634,5285,0.677375,8.068958,9.083493


,intensidad_principal,educacion_principal,n,n_ingreso_faltante,pct_ingreso_faltante
0,0.0,0.0,37940,6793,17.904586
1,0.0,1.0,24653,4187,16.983734
2,0.0,<NA>,456,85,18.640351
3,1.0,0.0,9503,1640,17.257708
4,1.0,1.0,6328,1043,16.482301
5,1.0,<NA>,106,16,15.09434
6,<NA>,0.0,753,127,16.86587
7,<NA>,1.0,556,97,17.446043
8,<NA>,<NA>,5,1,20.0


,grupo_edad,rho_spearman,p_valor,n_completo
0,10-17,-0.052733,2.564367e-01,465
1,18-24,0.035883,2.116136e-04,10655
2,25-34,-0.066050,4.266571e-19,18241
3,35-44,-0.106781,2.438059e-38,14620
4,45-54,-0.112322,3.751855e-34,11696
5,55-64,-0.097192,3.473603e-17,7489
6,65 o más,-0.111399,3.744254e-10,3145


,ANO4,TRIMESTRE,cluster_final,n,n_tres_componentes,pct_tres_componentes
0,2023,4,1,2601,2147,82.545175
1,2023,4,2,2302,1891,82.14596
2,2023,4,3,712,644,90.449438
3,2023,4,4,1152,1035,89.84375
4,2023,4,5,2292,1959,85.471204
5,2023,4,<NA>,55,39,70.909091
6,2024,1,1,2442,1941,79.484029
7,2024,1,2,2235,1779,79.597315
8,2024,1,3,686,540,78.717201
9,2024,1,4,1084,931,85.885609


Análisis de solapamiento completado. No se construyó un IVLR agregado.


## Construcción preliminar del IVLR continuo

Se construyen versiones preliminares y continuas del IVLR. Los pesos son normativos, transparentes y definidos antes de utilizar las transiciones laborales. No representan coeficientes estimados ni relaciones causales.

No se redistribuyen pesos cuando falta un componente. El IVLR básico se calcula únicamente cuando intensidad y educación están observadas. El IVLR ampliado y la sensibilidad de pesos iguales se calculan únicamente cuando los tres componentes están observados.

Todos los índices se orientan de 0 a 1, donde valores más altos representan mayor vulnerabilidad laboral relativa.

In [23]:
# Construcción preliminar de versiones continuas del IVLR.
# No utiliza información de t+1 ni redistribuye pesos.

COMP_INGRESO = "componente_ingreso"
COMP_INTENSIDAD = "intensidad_principal"
COMP_EDUCACION = "educacion_principal"


# ----------------------------------------------------------
# 1. Máscaras de disponibilidad
# ----------------------------------------------------------

mask_ivlr_basico = (
    base_ivlr[COMP_INTENSIDAD].notna()
    & base_ivlr[COMP_EDUCACION].notna()
)

mask_ivlr_ampliado = (
    base_ivlr[COMP_INGRESO].notna()
    & base_ivlr[COMP_INTENSIDAD].notna()
    & base_ivlr[COMP_EDUCACION].notna()
)


# ----------------------------------------------------------
# 2. IVLR básico
# Intensidad 70 %, educación 30 %
# ----------------------------------------------------------

base_ivlr["ivlr_basico"] = pd.Series(
    pd.NA,
    index=base_ivlr.index,
    dtype="Float64",
)

base_ivlr.loc[
    mask_ivlr_basico,
    "ivlr_basico",
] = (
    0.70
    * base_ivlr.loc[
        mask_ivlr_basico,
        COMP_INTENSIDAD,
    ]
    + 0.30
    * base_ivlr.loc[
        mask_ivlr_basico,
        COMP_EDUCACION,
    ]
)


# ----------------------------------------------------------
# 3. IVLR ampliado
# Ingreso 45 %, intensidad 35 %, educación 20 %
# ----------------------------------------------------------

base_ivlr["ivlr_ampliado"] = pd.Series(
    pd.NA,
    index=base_ivlr.index,
    dtype="Float64",
)

base_ivlr.loc[
    mask_ivlr_ampliado,
    "ivlr_ampliado",
] = (
    0.45
    * base_ivlr.loc[
        mask_ivlr_ampliado,
        COMP_INGRESO,
    ]
    + 0.35
    * base_ivlr.loc[
        mask_ivlr_ampliado,
        COMP_INTENSIDAD,
    ]
    + 0.20
    * base_ivlr.loc[
        mask_ivlr_ampliado,
        COMP_EDUCACION,
    ]
)


# ----------------------------------------------------------
# 4. Sensibilidad con pesos iguales
# ----------------------------------------------------------

base_ivlr["ivlr_ampliado_pesos_iguales"] = pd.Series(
    pd.NA,
    index=base_ivlr.index,
    dtype="Float64",
)

base_ivlr.loc[
    mask_ivlr_ampliado,
    "ivlr_ampliado_pesos_iguales",
] = (
    base_ivlr.loc[
        mask_ivlr_ampliado,
        [
            COMP_INGRESO,
            COMP_INTENSIDAD,
            COMP_EDUCACION,
        ],
    ]
    .mean(axis=1)
    .astype(float)
)


# ----------------------------------------------------------
# 5. Controles
# ----------------------------------------------------------

variables_ivlr = [
    "ivlr_basico",
    "ivlr_ampliado",
    "ivlr_ampliado_pesos_iguales",
]

for variable in variables_ivlr:
    assert (
        base_ivlr[variable]
        .dropna()
        .between(0, 1)
        .all()
    ), f"{variable} contiene valores fuera de [0, 1]"

assert (
    base_ivlr["ivlr_basico"].notna().sum()
    == mask_ivlr_basico.sum()
)

assert (
    base_ivlr["ivlr_ampliado"].notna().sum()
    == mask_ivlr_ampliado.sum()
)

assert (
    base_ivlr[
        "ivlr_ampliado_pesos_iguales"
    ].notna().sum()
    == mask_ivlr_ampliado.sum()
)

assert int(mask_ivlr_ampliado.sum()) == 64761


# ----------------------------------------------------------
# 6. Auditoría descriptiva
# ----------------------------------------------------------

def resumir_ivlr(variable, especificacion, pesos):
    serie = base_ivlr[variable]

    return {
        "variable": variable,
        "especificacion": especificacion,
        "pesos": pesos,
        "n_total": len(serie),
        "n_observado": int(serie.notna().sum()),
        "n_faltante": int(serie.isna().sum()),
        "pct_observado": float(
            100 * serie.notna().mean()
        ),
        "minimo": float(serie.min(skipna=True)),
        "p25": float(serie.quantile(0.25)),
        "mediana": float(serie.median(skipna=True)),
        "media": float(serie.mean(skipna=True)),
        "p75": float(serie.quantile(0.75)),
        "maximo": float(serie.max(skipna=True)),
        "estado": "especificación preliminar",
    }


auditoria_ivlr = pd.DataFrame([
    resumir_ivlr(
        "ivlr_basico",
        "básico",
        "intensidad=0.70; educación=0.30",
    ),
    resumir_ivlr(
        "ivlr_ampliado",
        "ampliado",
        "ingreso=0.45; intensidad=0.35; educación=0.20",
    ),
    resumir_ivlr(
        "ivlr_ampliado_pesos_iguales",
        "sensibilidad",
        "ingreso=1/3; intensidad=1/3; educación=1/3",
    ),
])

display(auditoria_ivlr)


# ----------------------------------------------------------
# 7. Comparación entre especificaciones
# ----------------------------------------------------------

comparacion_ivlr = (
    base_ivlr[
        [
            "ivlr_basico",
            "ivlr_ampliado",
            "ivlr_ampliado_pesos_iguales",
        ]
    ]
    .corr(method="spearman")
    .reset_index()
    .rename(columns={"index": "variable"})
)

display(comparacion_ivlr)


# ----------------------------------------------------------
# 8. Distribución por cluster
# Solo contexto descriptivo
# ----------------------------------------------------------

ivlr_por_cluster = (
    base_ivlr
    .groupby(
        [
            "cluster_final",
            "cluster_etiqueta",
        ],
        dropna=False,
        observed=True,
    )
    .agg(
        n=("id_persona", "size"),
        n_ivlr_basico=(
            "ivlr_basico",
            lambda x: x.notna().sum(),
        ),
        media_ivlr_basico=(
            "ivlr_basico",
            "mean",
        ),
        mediana_ivlr_basico=(
            "ivlr_basico",
            "median",
        ),
        n_ivlr_ampliado=(
            "ivlr_ampliado",
            lambda x: x.notna().sum(),
        ),
        media_ivlr_ampliado=(
            "ivlr_ampliado",
            "mean",
        ),
        mediana_ivlr_ampliado=(
            "ivlr_ampliado",
            "median",
        ),
    )
    .reset_index()
)

display(ivlr_por_cluster)


# ----------------------------------------------------------
# 9. Distribución por periodo
# ----------------------------------------------------------

ivlr_por_periodo = (
    base_ivlr
    .groupby(
        ["ANO4", "TRIMESTRE"],
        dropna=False,
    )
    .agg(
        n=("id_persona", "size"),
        n_ivlr_basico=(
            "ivlr_basico",
            lambda x: x.notna().sum(),
        ),
        media_ivlr_basico=(
            "ivlr_basico",
            "mean",
        ),
        n_ivlr_ampliado=(
            "ivlr_ampliado",
            lambda x: x.notna().sum(),
        ),
        media_ivlr_ampliado=(
            "ivlr_ampliado",
            "mean",
        ),
    )
    .reset_index()
)

display(ivlr_por_periodo)


# ----------------------------------------------------------
# 10. Exportaciones nuevas
# ----------------------------------------------------------

auditoria_ivlr.to_csv(
    OUT_DIR / "auditoria_ivlr_preliminar.csv",
    index=False,
    encoding="utf-8-sig",
)

comparacion_ivlr.to_csv(
    OUT_DIR / "comparacion_especificaciones_ivlr.csv",
    index=False,
    encoding="utf-8-sig",
)

ivlr_por_cluster.to_csv(
    OUT_DIR / "ivlr_preliminar_por_cluster.csv",
    index=False,
    encoding="utf-8-sig",
)

ivlr_por_periodo.to_csv(
    OUT_DIR / "ivlr_preliminar_por_periodo.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Versiones preliminares del IVLR construidas. "
    "No se definieron categorías de vulnerabilidad "
    "ni se utilizaron transiciones."
)

,variable,especificacion,pesos,n_total,n_observado,n_faltante,pct_observado,minimo,p25,mediana,media,p75,maximo,estado
0,ivlr_basico,básico,intensidad=0.70; educación=0.30,80300,78424,1876,97.663761,0.0,0.000000,0.300000,0.259818,0.300000,1.000000,especificación preliminar
1,ivlr_ampliado,ampliado,ingreso=0.45; intensidad=0.35; educación=0.20,80300,64761,15539,80.648817,0.0,0.197855,0.339703,0.374424,0.530184,0.991067,especificación preliminar
2,ivlr_ampliado_pesos_iguales,sensibilidad,ingreso=1/3; intensidad=1/3; educación=1/3,80300,64761,15539,80.648817,0.0,0.146559,0.353155,0.366025,0.548406,0.993383,especificación preliminar


,variable,ivlr_basico,ivlr_ampliado,ivlr_ampliado_pesos_iguales
0,ivlr_basico,1.000000,0.806482,0.890966
1,ivlr_ampliado,0.806482,1.000000,0.958183
2,ivlr_ampliado_pesos_iguales,0.890966,0.958183,1.000000


,cluster_final,cluster_etiqueta,n,n_ivlr_basico,media_ivlr_basico,mediana_ivlr_basico,n_ivlr_ampliado,media_ivlr_ampliado,mediana_ivlr_ampliado
0,1,Cuentapropismo predominantemente unipersonal,24571,24125,0.332402,0.3,19205,0.424652,0.391143
1,2,Asalariados informales jóvenes vinculados al c...,19286,18837,0.19587,0.0,15109,0.307137,0.269704
2,3,Asalariados informales en establecimientos gra...,5783,5482,0.172765,0.0,4951,0.289597,0.256096
3,4,Asalariados informales en hogares particulares,9867,9686,0.368811,0.3,8523,0.514079,0.493362
4,5,Inserciones informales mixtas en pequeños esta...,20244,19753,0.204906,0.0,16597,0.332701,0.304724
5,<NA>,<NA>,549,541,0.185397,0.0,376,0.305728,0.269704


,ANO4,TRIMESTRE,n,n_ivlr_basico,media_ivlr_basico,n_ivlr_ampliado,media_ivlr_ampliado
0,2023,4,9114,8974,0.253009,7715,0.371771
1,2024,1,8652,8230,0.261057,7001,0.374226
2,2024,2,8951,8746,0.268043,7411,0.377414
3,2024,3,9167,8998,0.269293,7406,0.379256
4,2024,4,9109,8949,0.262867,7206,0.375911
5,2025,1,8656,8325,0.250294,6767,0.368961
6,2025,2,8989,8856,0.25227,7274,0.371287
7,2025,3,8916,8734,0.261243,7041,0.375735
8,2025,4,8746,8612,0.259835,6940,0.37496


Versiones preliminares del IVLR construidas. No se definieron categorías de vulnerabilidad ni se utilizaron transiciones.


## 6. Auditoría de componentes

Se genera una tabla resumen por componente con estadísticos básicos y estado metodológico. También se calcula la cobertura por año-trimestre, CAT_OCUP y cluster_final.

In [11]:
# %%
# 6.1 Tabla resumen por componente (sin media/mediana para educación)
def resumen_componente(variable, dimension, serie, ordinal=False):
    obs = int(serie.notna().sum())
    falt = int(serie.isna().sum())

    resumen = {
        "variable": variable,
        "dimension": dimension,
        "universo": "empleo informal",
        "n_total": len(serie),
        "n_observado": obs,
        "n_faltante": falt,
        "porcentaje_observado": 100 * obs / len(serie),
        "minimo": serie.min(skipna=True),
        "maximo": serie.max(skipna=True),
        "media": np.nan,
        "mediana": np.nan,
    }

    if not ordinal:
        resumen["media"] = serie.mean(skipna=True)
        resumen["mediana"] = serie.median(skipna=True)

    return resumen

componentes = [
    (
        "componente_ingreso",
        "Ingreso relativo",
        base_ivlr["componente_ingreso"],
        False,
    ),
    (
        "int_subocupado",
        "Intensidad - Subocupado",
        base_ivlr["int_subocupado"].astype(float),
        False,
    ),
    (
        "int_ocupado_pleno",
        "Intensidad - Ocupado pleno",
        base_ivlr["int_ocupado_pleno"].astype(float),
        False,
    ),
    (
        "int_sobreocupado",
        "Intensidad - Sobreocupado",
        base_ivlr["int_sobreocupado"].astype(float),
        False,
    ),
    (
        "int_no_trabajo",
        "Intensidad - No trabajó",
        base_ivlr["int_no_trabajo"].astype(float),
        False,
    ),
    (
        "educacion_codigo_original",
        "Educación",
        base_ivlr["educacion_codigo_numerico"],
        True,
    ),
]

auditoria_componentes = pd.DataFrame([
    resumen_componente(variable, dimension, serie, ordinal)
    for variable, dimension, serie, ordinal in componentes
])

# Añadir estado metodológico
estados = {
    "componente_ingreso": "ingreso relativo construido",
    "int_subocupado": "categorías de intensidad pendientes de ponderación",
    "int_ocupado_pleno": "categorías de intensidad pendientes de ponderación",
    "int_sobreocupado": "categorías de intensidad pendientes de ponderación",
    "int_no_trabajo": "categorías de intensidad pendientes de ponderación",
    "educacion_codigo_original": "codificación educativa pendiente de decisión conceptual",
}
auditoria_componentes["estado_metodologico"] = auditoria_componentes["variable"].map(estados)

display(auditoria_componentes)
auditoria_componentes.to_csv(OUT_DIR / "auditoria_componentes_ivlr.csv", index=False, encoding="utf-8-sig")

# 6.2 Cobertura por año-trimestre (para componente_ingreso)
cobertura_periodo = base_ivlr.groupby(["ANO4", "TRIMESTRE"], dropna=False).agg(
    n_total=("componente_ingreso", "size"),
    n_observado=("componente_ingreso", lambda x: x.notna().sum()),
    pct_observado=("componente_ingreso", lambda x: 100 * x.notna().mean()),
    n_faltante=("componente_ingreso", lambda x: x.isna().sum()),
).reset_index()
cobertura_periodo["dimension"] = "Ingreso relativo"
display(cobertura_periodo)
cobertura_periodo.to_csv(OUT_DIR / "cobertura_componentes_por_periodo.csv", index=False, encoding="utf-8-sig")

# 6.3 Cobertura por CAT_OCUP (para componente_ingreso y educación)
def cobertura_por_cat(col, label):
    return base_ivlr.groupby("CAT_OCUP", dropna=False).agg(
        n_total=(col, "size"),
        n_observado=(col, lambda x: x.notna().sum()),
        pct_observado=(col, lambda x: 100 * x.notna().mean()),
    ).reset_index().assign(dimension=label)

cat_ingreso = cobertura_por_cat("componente_ingreso", "Ingreso relativo")
cat_educ = cobertura_por_cat("educacion_codigo_original", "Educación")
cobertura_cat = pd.concat([cat_ingreso, cat_educ], ignore_index=True)
display(cobertura_cat)
cobertura_cat.to_csv(OUT_DIR / "cobertura_componentes_por_cat_ocup.csv", index=False, encoding="utf-8-sig")

# 6.4 Cobertura por cluster_final (contexto descriptivo)
cobertura_cluster = base_ivlr.groupby("cluster_final", dropna=False).agg(
    n_total=("componente_ingreso", "size"),
    n_obs_ingreso=("componente_ingreso", lambda x: x.notna().sum()),
    pct_obs_ingreso=("componente_ingreso", lambda x: 100 * x.notna().mean()),
    n_obs_educ=("educacion_codigo_original", lambda x: x.notna().sum()),
    pct_obs_educ=("educacion_codigo_original", lambda x: 100 * x.notna().mean()),
).reset_index()
display(cobertura_cluster)
cobertura_cluster.to_csv(OUT_DIR / "cobertura_componentes_por_cluster.csv", index=False, encoding="utf-8-sig")

,variable,dimension,universo,n_total,n_observado,n_faltante,porcentaje_observado,minimo,maximo,media,mediana,estado_metodologico
0,componente_ingreso,Ingreso relativo,empleo informal,80300,66311,13989,82.579078,0.0,0.980148,0.499932,0.514388,ingreso relativo construido
1,int_subocupado,Intensidad - Subocupado,empleo informal,80300,80300,0,100.000000,0.0,1.000000,0.198468,0.000000,categorías de intensidad pendientes de pondera...
2,int_ocupado_pleno,Intensidad - Ocupado pleno,empleo informal,80300,80300,0,100.000000,0.0,1.000000,0.530386,1.000000,categorías de intensidad pendientes de pondera...
3,int_sobreocupado,Intensidad - Sobreocupado,empleo informal,80300,80300,0,100.000000,0.0,1.000000,0.254782,0.000000,categorías de intensidad pendientes de pondera...
4,int_no_trabajo,Intensidad - No trabajó,empleo informal,80300,80300,0,100.000000,0.0,1.000000,0.016364,0.000000,categorías de intensidad pendientes de pondera...
5,educacion_codigo_original,Educación,empleo informal,80300,80300,0,100.000000,1.0,7.000000,NaN,NaN,codificación educativa pendiente de decisión c...


,ANO4,TRIMESTRE,n_total,n_observado,pct_observado,n_faltante,dimension
0,2023,4,9114,7833,85.944700,1281,Ingreso relativo
1,2024,1,8652,7366,85.136385,1286,Ingreso relativo
2,2024,2,8951,7576,84.638588,1375,Ingreso relativo
3,2024,3,9167,7537,82.218828,1630,Ingreso relativo
4,2024,4,9109,7335,80.524756,1774,Ingreso relativo
5,2025,1,8656,7046,81.400185,1610,Ingreso relativo
6,2025,2,8989,7383,82.133719,1606,Ingreso relativo
7,2025,3,8916,7185,80.585464,1731,Ingreso relativo
8,2025,4,8746,7050,80.608278,1696,Ingreso relativo


,CAT_OCUP,n_total,n_observado,pct_observado,dimension
0,1,1772,1439,81.207675,Ingreso relativo
1,2,28245,22403,79.316693,Ingreso relativo
2,3,49514,41700,84.218605,Ingreso relativo
3,4,769,769,100.000000,Ingreso relativo
4,1,1772,1772,100.000000,Educación
5,2,28245,28245,100.000000,Educación
6,3,49514,49514,100.000000,Educación
7,4,769,769,100.000000,Educación


,cluster_final,n_total,n_obs_ingreso,pct_obs_ingreso,n_obs_educ,pct_obs_educ
0,1,24571,19542,79.532783,24571,100.0
1,2,19286,15487,80.301773,19286,100.0
2,3,5783,5221,90.281861,5783,100.0
3,4,9867,8685,88.020675,9867,100.0
4,5,20244,16995,83.950800,20244,100.0
5,<NA>,549,381,69.398907,549,100.0


## 7. Exportación de la base de componentes

Se guarda la base completa (con todos los componentes) en Parquet y CSV, manteniendo las 80.300 filas originales.

In [18]:
# %%
# Seleccionar columnas finales
cols_exportar = [
    "id_persona", "ANO4", "TRIMESTRE", "PONDERA",
    "CH06", "edad", "grupo_edad",
    "CAT_OCUP", "NIVEL_ED", "INTENSI",
    "log_P21_real", "P21_real", "percentil_ingreso", "componente_ingreso",
    "int_subocupado", "int_ocupado_pleno", "int_sobreocupado", "int_no_trabajo",
    "educacion_codigo_original",
    "educacion_etiqueta",
    "cat_ocup_etiqueta",
    "cluster_etiqueta",
    "cluster_final"
]
base_export = base_ivlr[cols_exportar].copy()

# Verificar que se conservan todas las filas
assert len(base_export) == 80300

# Guardar Parquet con PyArrow
table = pa.Table.from_pandas(base_export, preserve_index=False)
pq.write_table(table, OUT_DIR / "base_componentes_ivlr.parquet")

# Verificar el archivo Parquet
tabla_verificacion = pq.read_table(OUT_DIR / "base_componentes_ivlr.parquet")
assert tabla_verificacion.num_rows == 80300
assert tabla_verificacion.num_columns == len(base_export.columns)

# Guardar CSV
base_export.to_csv(OUT_DIR / "base_componentes_ivlr.csv", index=False, encoding="utf-8-sig")

print("Exportación completada y verificada.")

Exportación completada y verificada.


In [19]:
# %%
# Versiones del entorno (incluyendo fuente de datos)
versiones = pd.DataFrame({
    "componente": [
        "Python",
        "intérprete",
        "pandas",
        "numpy",
        "pyarrow",
        "fuente de datos utilizada",
    ],
    "version": [
        sys.version.split()[0],
        sys.executable,
        pd.__version__,
        np.__version__,
        pa.__version__,
        fuente_datos,
    ],
})
versiones.to_csv(OUT_DIR / "versiones_entorno_ivlr.csv", index=False, encoding="utf-8-sig")
print("Versiones del entorno guardadas.")

Versiones del entorno guardadas.


In [20]:
# %%
# Conclusiones
conclusiones = (
    "Esta etapa construye y audita componentes candidatos del IVLR, "
    "pero no define todavía un índice agregado. "
    "El componente económico representa la posición relativa del "
    "ingreso dentro de cada año-trimestre. "
    "Las categorías de intensidad laboral y educación se conservan "
    "sin ponderación hasta cerrar sus reglas conceptuales. "
    "La estabilidad laboral permanece pendiente debido a la necesidad "
    "de armonizar universos ocupacionales y revisar el tratamiento "
    "específico del personal de casas particulares.\n"
)

with open(OUT_DIR / "conclusiones_componentes_ivlr.txt", "w", encoding="utf-8") as archivo:
    archivo.write(conclusiones)

print("Conclusiones guardadas.")

Conclusiones guardadas.
